# Evolución de la propuesta multiagente

El dashboard compara las etapas disponibles y separa entrenamiento de evaluación. Ejecuta las celdas en orden. Al cargar, lee los CSV; solo vuelve a simular cuando se pulsa **Regenerar telemetría**.

1. **Baseline + A\***: trabajo por franjas y decisiones individuales.
2. **Improved + A\* + Blackboard**: coordinación, asignación de tareas y prevención de conflictos.
3. **Improved + A\* + Blackboard + Q-learning**: añade la política de reabastecimiento de semillas. Q-learning no elige celdas ni decide el reabastecimiento de combustible.

OpenClaw y MCP son posibilidades futuras; no son implementaciones terminadas.

## 1. Preparar el proyecto y revisar las fuentes

In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import Markdown, display

start = Path.cwd().resolve()
REPO_ROOT = next((p for p in (start, *start.parents)
                  if (p / "src" / "model.py").is_file()
                  and (p / "experiments" / "stage4.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Abre este notebook desde el repositorio Reto_Multiagentes.")

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from experiments.dashboard import show_dashboard, show_episode_replay
print(f"Repositorio: {REPO_ROOT}")

## 2. Comparar escenarios

El selector permite elegir la cohorte, densidad, modo de inicio (cuando el CSV lo declara), semilla y métrica. Los puntos muestran episodios; los rombos muestran medianas. Las diferencias pareadas se calculan por densidad y semilla, añadiendo las claves opcionales de inicio/escenario/configuración cuando están guardadas. Cada carpeta Q se mantiene aislada.

In [ ]:
show_dashboard(REPO_ROOT)

## 3. Evolución temporal y espacial

Los CSV de resúmenes no guardan trayectorias por tick. La siguiente celda ofrece reproducir un episodio seleccionado en memoria para registrar cobertura, mapa sembrado y rutas. Si se elige Q-learning, carga la política de esa carpeta. La reproducción usa el checkout actual y puede diferir de un CSV antiguo si el código o sus parámetros cambiaron.

In [ ]:
show_episode_replay(REPO_ROOT)

## 4. Reproducibilidad y límites

- La fase `train` de Q-learning queda en una tabla aparte; solo `eval` entra a comparaciones. Los resultados Q se emparejan dentro de su propio CSV y no se mezclan con etapa 4 ni con otras carpetas Q.
- Los episodios incompletos se identifican por fila. Sus ticks pueden estar censurados en el límite; el tamaño de muestra para pares completos también se informa.
- Los resultados bajo `results/` están ignorados por Git. Comparte `results/stage4/episodes.csv` y cada carpeta `results/qlearning*/` que quieras analizar. Conserva sus cohortes separadas.
- El formato admite columnas futuras `start_mode` (o `initial_positions_mode`), `start_seed` (o `initial_positions_seed`), posiciones iniciales serializadas, `scenario_id` y `config_hash` y `parameter_hash`. Si no existen, el dashboard lo señala y no infiere el modo de inicio.
- Para comandos de generación, archivos fuente y limitaciones, consulta [`DASHBOARD.md`](../DASHBOARD.md).